In [ ]:
import numpy as np

from myopt import data_structure as ds
from myopt import bind_optimizer, get_optimizer, optimize
from myopt import matching_pursuit, get_matching_pursuit, bind_matching_pursuit
from myopt import mcco_modeling

# from problems_generator import compressive_sensing_pb as cs_pb
# from problems_generator import compressible_opt_pb as co_pb
# from functools import partial

In [5]:
#example
number_spins = 7

#abstract
spectrum_bin = [
    [0,0,0,0,0,1,0],
    [0,1,1,1,0,1,0],
    [0,0,1,1,1,1,0],
    [1,1,1,1,0,0,1]
]
spectrum_pos = [ds.dit_string_to_integer(s) for s in spectrum_bin]
spectrum_val = [-0.5,1.8,-0.3,1.5]

#explicit
full_spectrum = np.zeros((2**number_spins,))
full_spectrum[spectrum_pos] = spectrum_val


#Define marginals and sketch
sketch_builder = sketchs.ConstraintSketch
interaction_size = 4
constraints = sketch_builder.build_nearest_neighbors_sketch(number_spins, interaction_size, 2)
y = sketch_builder.compute_marginal((spectrum_bin, spectrum_val), constraints)

In [4]:
#Abstract optimization

#NN spin chain (Default)
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size))

#Dual annealing
opti = get_optimizer("dual_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

#Simulated annealing
opti = get_optimizer("simulated_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[ 58.        2.175  ]
 [121.        1.40625]]
[[ 58.        2.175  ]
 [121.        1.40625]]
[[ 58.        2.175  ]
 [121.        1.40625]]


In [17]:
#Explicit methods

#Define marginals and sketch explicitly
sketch_builder = sketchs.ExplicitSketch
sketch = sketch_builder.build_nearest_neighbors_sketch(number_spins,interaction_size)
y = sketch_builder.compute_marginal(full_spectrum, sketch)

#Brute force (Default)
print(matching_pursuit("explicit", y, sketch, 2))

sketch = sketch_builder.random_sketch(number_spins, 64)
y = sketch_builder.compute_marginal(full_spectrum, sketch)
print(matching_pursuit("explicit", y, sketch, 2))

[[ 58.        2.175  ]
 [121.        1.40625]]
[[ 58.           1.73634171]
 [121.           1.5448792 ]]


In [18]:
#example 2
number_spins = 6

#abstract
spectrum_bin = [
    [0,0,0,0,0,0],
    [0,1,0,1,0,1],
    [0,0,1,0,0,1],
    [1,1,1,1,1,1]
]
spectrum_pos = [ds.dit_string_to_integer(s) for s in spectrum_bin]
spectrum_val = [0.5,5,3,2]

#explicit
full_spectrum = np.zeros((2**number_spins,))
full_spectrum[spectrum_pos] = spectrum_val

#Define marginals and sketch
sketch_builder = sketchs.ConstraintSketch
interaction_size = 2
constraints = sketch_builder.build_nearest_neighbors_sketch(number_spins, interaction_size, 2)
y = sketch_builder.compute_marginal((spectrum_bin, spectrum_val), constraints)

In [20]:
#Classical methods

#NN_spin_chain
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size))

#Dual annealing
opti = get_optimizer("dual_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

#Simulated annealing
opti = get_optimizer("simulated_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[21.    5.6 ]
 [ 9.    3.08]]
[[21.    5.6 ]
 [ 9.    3.08]]
[[21.    5.6 ]
 [57.    2.58]]


In [21]:
#Quantum methods

#Digital annealing
opti = get_optimizer("digital_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

#QAOA
opti = bind_optimizer("qaoa", number_shots=4096)
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[21.    5.6 ]
 [ 9.    3.08]]
[[21.    5.6 ]
 [ 9.    3.08]]


In [56]:
#Example 3
# number_spins = 12
# rules_reward = co_pb.generate_rules_and_rewards(2**number_spins, num_rules=3)

# number_spins = 12
# rules_reward = {(0, 0, 0, 0): 18, (1, 1, 0, 1): 5, (1, 1, 1, 1): 18}

number_spins = 16
rules_reward = {(0, 1, 1, 0): 2, (1, 1, 0, 1): 15, (0, 0, 1, 0): 1}
print(rules_reward)

{(0, 1, 1, 0): 2, (1, 1, 0, 1): 15, (0, 0, 1, 0): 1}


In [57]:
import matplotlib.pyplot as plt

full_spectrum = co_pb.compute_full_spectrum(number_spins, rules_reward)
print(np.where(full_spectrum == np.max(full_spectrum)))
# plt.bar(range(2**number_spins), full_spectrum)
# plt.show()

(array([56173]),)


In [ ]:
from mcco_workflow import solve_via_mcco

objective = partial(co_pb.estimate_cost, rules_rewards=rules_reward)

result = solve_via_mcco(
    objective_function=objective,
    number_samples=5000,
    dit_string_length=number_spins,
    interaction_size=4,
    iteration_number=5,
    thereshold_parameter="Auto",
    optimizer=get_optimizer("spin_chain_nn_max"),
)

print("spectrum_pos:", result["spectrum_pos"])
print("spectrum_val:", result["spectrum_val"])
print("spectrum_bin:", result["spectrum_bin"])
print("solution:\n", result["solution"])

spectrum_pos: (730, 886, 1261, 1677, 1716, 1749, 1759, 2893, 3381, 3433, 3501, 3513, 3545, 4539, 4970, 4971, 4972, 5047, 5549, 5558, 5841, 5843, 5851, 5855, 5870, 6589, 6710, 6762, 6837, 6875, 6965, 6987, 7023, 7039, 7387, 7603, 7661, 9133, 9143, 9453, 9655, 9837, 9948, 10157, 10167, 10971, 11189, 11190, 11245, 11626, 11629, 11639, 11677, 11685, 11686, 11698, 11705, 11710, 11738, 11740, 11742, 11743, 11988, 12725, 13163, 13179, 13236, 13418, 13485, 13734, 13750, 13914, 13931, 13960, 13963, 14017, 14023, 14031, 14047, 14072, 14106, 14168, 14181, 14189, 14257, 14317, 15067, 15130, 15208, 15287, 15322, 15341, 15725, 15757, 15770, 15837, 16089, 16237, 17243, 17259, 17260, 18075, 18093, 18132, 18868, 19308, 19325, 19803, 19903, 19922, 20205, 21933, 21941, 21943, 22224, 22225, 22746, 22957, 23383, 23386, 23399, 23408, 23412, 23474, 23513, 23532, 24026, 24285, 24301, 24429, 24795, 24797, 25455, 26042, 26077, 26335, 26345, 26362, 26459, 26473, 26477, 26605, 26678, 26717, 26805, 26807, 26813, 2

In [62]:

# Benchmark : sur N problèmes générés avec co_pb.generate_rules_and_rewards (taille 2**16),
# combien de fois solve_via_mcco retrouve-t-il le maximum global ?

import time

number_spins_bench = 16
n_problems = 100
num_rules = 3
success_count = 0
timings = []

for i in range(n_problems):
    # Générer un problème aléatoire
    rules_reward_bench = co_pb.generate_rules_and_rewards(2**number_spins_bench, num_rules=num_rules)

    # Calculer le vrai maximum
    full_spectrum_bench = co_pb.compute_full_spectrum(number_spins_bench, rules_reward_bench)
    true_max_positions = set(np.where(full_spectrum_bench == np.max(full_spectrum_bench))[0].tolist())

    # Résoudre via MCCO
    objective_bench = partial(co_pb.estimate_cost, rules_rewards=rules_reward_bench)
    t0 = time.time()
    result_bench = solve_via_mcco(
        objective_function=objective_bench,
        number_samples=5000,
        dit_string_length=number_spins_bench,
        interaction_size=4,
        iteration_number=5,
        thereshold_parameter="Auto",
        optimizer=get_optimizer("spin_chain_nn_max"),
    )
    elapsed = time.time() - t0
    timings.append(elapsed)

    # Vérifier si le maximum est trouvé dans la 1ère colonne de solution
    found_positions = set(result_bench["solution"][:, 0].tolist())
    hit = bool(true_max_positions & found_positions)
    success_count += int(hit)

    print(f"[{i+1:02d}/{n_problems}] {'✓ HIT' if hit else '✗ MISS'} | {elapsed:.1f}s")

print(f"\n=== Résultats benchmark ===")
print(f"Taille du problème : 2^{number_spins_bench} = {2**number_spins_bench}")
print(f"Succès : {success_count}/{n_problems}  ({100*success_count/n_problems:.0f}%)")
print(f"Temps moyen par problème : {np.mean(timings):.2f}s  (total : {np.sum(timings):.1f}s)")


[01/100] ✗ MISS | 0.6s
[02/100] ✓ HIT | 0.6s
[03/100] ✓ HIT | 0.6s
[04/100] ✓ HIT | 0.6s
[05/100] ✗ MISS | 0.6s
[06/100] ✓ HIT | 0.6s
[07/100] ✓ HIT | 0.6s
[08/100] ✓ HIT | 0.6s
[09/100] ✓ HIT | 0.6s
[10/100] ✓ HIT | 0.6s
[11/100] ✓ HIT | 0.6s
[12/100] ✓ HIT | 0.6s
[13/100] ✓ HIT | 0.6s
[14/100] ✓ HIT | 0.6s
[15/100] ✓ HIT | 0.6s
[16/100] ✓ HIT | 0.6s
[17/100] ✓ HIT | 0.8s
[18/100] ✗ MISS | 0.6s
[19/100] ✓ HIT | 0.6s
[20/100] ✗ MISS | 0.6s
[21/100] ✓ HIT | 0.6s
[22/100] ✓ HIT | 0.6s
[23/100] ✓ HIT | 0.6s
[24/100] ✓ HIT | 0.6s
[25/100] ✓ HIT | 0.6s
[26/100] ✓ HIT | 0.6s
[27/100] ✓ HIT | 0.6s
[28/100] ✓ HIT | 0.6s
[29/100] ✗ MISS | 0.6s
[30/100] ✗ MISS | 0.6s
[31/100] ✓ HIT | 0.6s
[32/100] ✗ MISS | 0.8s
[33/100] ✓ HIT | 0.6s
[34/100] ✗ MISS | 0.7s
[35/100] ✓ HIT | 0.6s
[36/100] ✓ HIT | 0.6s
[37/100] ✓ HIT | 0.6s
[38/100] ✓ HIT | 0.6s
[39/100] ✗ MISS | 0.6s
[40/100] ✓ HIT | 0.6s
[41/100] ✓ HIT | 0.6s
[42/100] ✓ HIT | 0.6s
[43/100] ✓ HIT | 0.6s
[44/100] ✗ MISS | 0.7s
[45/100] ✗ MISS | 0.6s